# Feature Engineering Pipeline

**Target**: `ratio = employees_count / office_present`
**Horizon**: 7 days (all features available at J-8 at latest)
**Working week**: Sunday→Thursday (DOW 6,0,1,2,3)
**Shift minimum**: 7 for every lag/rolling feature

In [1]:
import os, sys, warnings, json, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
import holidays as hol

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

SEED = 42
HORIZON = 7
REPO_ROOT = Path().resolve().parent
PROC_DIR  = REPO_ROOT / 'data' / 'processed'
# --- Accès aux modules src (menu canonique partagé entraînement/serving) ---
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.menu_optimization.menu_catalog_py import build_menu_features  # noqa: E402,F401

FIG_DIR   = REPO_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROC_DIR / 'real_clean.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Loaded {len(df)} rows  |  {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Columns: {list(df.columns)}")

Loaded 605 rows  |  2022-05-05 → 2024-12-31
Columns: ['Date', 'office_present', 'employees_count', 'entrees', 'plat_principal_1', 'plat_principal_2', 'temperature', 'temperature_felt', 'precipitation_mm', 'precipitation_type', 'wind_gust_kmh', 'wind_speed_kmh', 'cloud_cover_pct', 'weather_conditions', 'dow', 'is_working_day', 'ratio', 'ratio_capped']


---
## Section 1 — Calendar Features

In [2]:
min_date = df['Date'].min()
df['day_of_month']   = df['Date'].dt.day
df['week_of_year']   = df['Date'].dt.isocalendar().week.astype(int)
df['month_n']        = df['Date'].dt.month
df['quarter']        = df['Date'].dt.quarter
df['year_n']         = df['Date'].dt.year
df['day_of_year']    = df['Date'].dt.dayofyear
df['days_in_month']  = df['Date'].dt.days_in_month

# DOW dummies (0=Mon ... 6=Sun)
df['dow'] = df['Date'].dt.dayofweek

# --- Algerian working-week flags ---
df['is_weekend']       = df['dow'].isin([4, 5]).astype(int)  # Fri, Sat
df['is_sun']           = (df['dow'] == 6).astype(int)
df['is_mon']           = (df['dow'] == 0).astype(int)
df['is_tue']           = (df['dow'] == 1).astype(int)
df['is_wed']           = (df['dow'] == 2).astype(int)
df['is_thu']           = (df['dow'] == 3).astype(int)
df['is_sunday_workday'] = df['is_sun']  # alias

# Fourier – annual seasonality (k = 1,2,3)
doy = df['day_of_year'].astype(float)
for k in range(1, 4):
    df[f'sin_year_{k}'] = np.sin(2 * np.pi * k * doy / 365.25)
    df[f'cos_year_{k}'] = np.cos(2 * np.pi * k * doy / 365.25)

# Fourier – weekly cycle (5-day Algerian week)
df['sin_week'] = np.sin(2 * np.pi * df['dow'] / 5)
df['cos_week'] = np.cos(2 * np.pi * df['dow'] / 5)

# Linear trend
df['trend_days'] = (df['Date'] - min_date).dt.days

# Month progress
df['month_progress'] = df['day_of_month'] / df['days_in_month']

# Week-of-month flags
df['week_of_month'] = (df['day_of_month'] - 1) // 7 + 1
df['is_week1_month'] = (df['week_of_month'] == 1).astype(int)
df['is_week4_month'] = (df['week_of_month'] >= 4).astype(int)

print(f"Calendar features added: {sum(1 for c in df.columns if c.startswith(('is_', 'sin_', 'cos_', 'trend', 'month_progress', 'week_of')))}")
print(f"Working-day rows: {(df['is_weekend']==0).sum()}  |  Weekend rows: {(df['is_weekend']==1).sum()}")

Calendar features added: 22
Working-day rows: 605  |  Weekend rows: 0


---
## Section 2 — Holiday Features

In [3]:
# --- French public holidays ----------------------------------------------------
fr_holidays = hol.France(years=range(2022, 2026))
fr_holiday_set = set(pd.Timestamp(d) for d, _ in fr_holidays.items())
df['is_fr_holiday'] = df['Date'].isin(fr_holiday_set).astype(int)

# --- Islamic holidays (hardcoded 2022-2025) -----------------------------------
islamic_holidays = pd.to_datetime([
    # Eid al-Fitr
    '2022-05-02', '2023-04-21', '2024-04-10', '2025-03-30',
    # Eid al-Adha
    '2022-07-09', '2023-06-28', '2024-06-16', '2025-06-07',
    # Mawlid
    '2022-10-08', '2023-09-27', '2024-09-15', '2025-09-05',
    # Islamic New Year
    '2022-07-30', '2023-07-19', '2024-07-07', '2025-06-26',
    # Ashura
    '2022-08-08', '2023-07-28', '2024-07-16', '2025-07-05',
])
islamic_set = set(islamic_holidays)
df['is_islamic_holiday'] = df['Date'].isin(islamic_set).astype(int)

# --- Algerian national holidays ------------------------------------------------
algerian_holidays = pd.to_datetime([
    '2022-07-05', '2023-07-05', '2024-07-05', '2025-07-05',  # Independence Day
    '2022-11-01', '2023-11-01', '2024-11-01', '2025-11-01',  # Revolution Day
])
algerian_set = set(algerian_holidays)
df['is_algerian_holiday'] = df['Date'].isin(algerian_set).astype(int)

# --- Combined holiday flag -----------------------------------------------------
df['is_any_holiday'] = ((df['is_fr_holiday'] | df['is_islamic_holiday'] | df['is_algerian_holiday'])).astype(int)

# --- days_to_next_holiday / days_since_last_holiday (capped 14) ----------------
holiday_mask = df['is_any_holiday'] == 1
holiday_dates = df.loc[holiday_mask, 'Date'].values

dt_next = np.full(len(df), 14.0)
dt_prev = np.full(len(df), 14.0)
for i, d in enumerate(df['Date'].values):
    future = holiday_dates[holiday_dates >= d]
    if len(future) > 0:
        dt_next[i] = min((future[0] - d) / np.timedelta64(1, 'D'), 14.0)
    past = holiday_dates[holiday_dates <= d]
    if len(past) > 0:
        dt_prev[i] = min((d - past[-1]) / np.timedelta64(1, 'D'), 14.0)

df['days_to_next_holiday'] = dt_next.astype(int)
df['days_since_last_holiday'] = dt_prev.astype(int)
df['near_holiday'] = ((df['days_to_next_holiday'] <= 2) | (df['days_since_last_holiday'] <= 2)).astype(int)

# --- bridge_day: working day between holiday and weekend -----------------------
df['is_workday'] = (df['is_weekend'] == 0) & (df['is_any_holiday'] == 0)
df['prev_is_holiday_or_wknd'] = df['is_any_holiday'].shift(1, fill_value=0) | df['is_weekend'].shift(1, fill_value=0)
df['next_is_holiday_or_wknd'] = df['is_any_holiday'].shift(-1, fill_value=0) | df['is_weekend'].shift(-1, fill_value=0)
df['bridge_day'] = (df['is_workday'] & df['prev_is_holiday_or_wknd'] & df['next_is_holiday_or_wknd']).astype(int)
df.drop(columns=['is_workday', 'prev_is_holiday_or_wknd', 'next_is_holiday_or_wknd'], inplace=True)

# --- Ramadan -------------------------------------------------------------------
ramadan_periods = [
    ('2022-04-02', '2022-05-01'),
    ('2023-03-22', '2023-04-20'),
    ('2024-03-11', '2024-04-09'),
    ('2025-02-28', '2025-03-29'),
]
df['is_ramadan'] = 0
df['ramadan_day'] = 0
for start, end in ramadan_periods:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    mask = (df['Date'] >= s) & (df['Date'] <= e)
    df.loc[mask, 'is_ramadan'] = 1
    df.loc[mask, 'ramadan_day'] = (df.loc[mask, 'Date'] - s).dt.days + 1

# --- Seasonal / calendar flags ------------------------------------------------
df['is_august']   = (df['month_n'] == 8).astype(int)
df['is_july']     = (df['month_n'] == 7).astype(int)
df['is_summer']   = df['month_n'].isin([6, 7, 8]).astype(int)
df['is_yearend']  = ((df['month_n'] == 12) & (df['day_of_month'] >= 22)).astype(int)
df['is_jan_start']= ((df['month_n'] == 1) & (df['day_of_month'] <= 10)).astype(int)

# --- Payday flags -------------------------------------------------------------
df['is_payday1'] = df['day_of_month'].isin(range(1, 6)).astype(int)
df['is_payday2'] = df['day_of_month'].isin(range(14, 17)).astype(int)
df['is_payday3'] = (df['day_of_month'] >= 26).astype(int)

holiday_feats = [c for c in df.columns if any(k in c for k in
    ['holiday', 'ramadan', 'august', 'july', 'summer', 'yearend', 'jan_start', 'payday', 'bridge', 'near_holiday', 'algerian'])]
print(f"Holiday features added: {len(holiday_feats)}")
print(f"  is_any_holiday=1 rows: {df['is_any_holiday'].sum()}")
print(f"  is_ramadan=1 rows:     {df['is_ramadan'].sum()}")
print(f"  bridge_day=1 rows:     {df['bridge_day'].sum()}")

Holiday features added: 18
  is_any_holiday=1 rows: 21
  is_ramadan=1 rows:     2
  bridge_day=1 rows:     0


---
## Section 3 — Office Presence Features

All lags and rolling windows use `shift(7)` minimum to respect the J-7 horizon.

In [4]:
# --- Lag features (shift >= 7) -------------------------------------------------
for lag in [7, 14, 21, 28]:
    df[f'op_lag_{lag}'] = df['office_present'].shift(lag)

# --- Rolling statistics (base shift=7, then rolling window) --------------------
base = df['office_present'].shift(7)
for w in [28]:  # 4 weeks
    df[f'op_roll_mean_{w}'] = base.rolling(w).mean()
    df[f'op_roll_std_{w}']  = base.rolling(w).std()
    df[f'op_roll_max_{w}']  = base.rolling(w).max()
    df[f'op_roll_min_{w}']  = base.rolling(w).min()
for w in [56]:  # 8 weeks
    df[f'op_roll_mean_{w}'] = base.rolling(w).mean()

# --- DOW-conditional stats (training-only: use data up to row i-1) -----------
op_dow_means = {}
op_dow_stds  = {}
op_dow_medians = {}
running_op = []   # will accumulate office_present values row by row

op_dow_mean_vals  = np.full(len(df), np.nan)
op_dow_std_vals   = np.full(len(df), np.nan)
op_dow_med_vals   = np.full(len(df), np.nan)

for i in range(len(df)):
    d = df.iloc[i]['dow']
    if i >= 7:  # at least one full week of history
        past = df.iloc[:i]
        past_dow = past[past['dow'] == d]['office_present'].dropna()
        if len(past_dow) >= 3:
            op_dow_mean_vals[i]  = past_dow.mean()
            op_dow_std_vals[i]   = past_dow.std()
            op_dow_med_vals[i]   = past_dow.median()

df['op_dow_mean']   = op_dow_mean_vals
df['op_dow_std']    = op_dow_std_vals
df['op_dow_median'] = op_dow_med_vals
df['op_dow_zscore'] = (df['office_present'] - df['op_dow_mean']) / (df['op_dow_std'] + 1)

# --- Annual zscore (global mean/std of office_present, no leakage) ------------
# Use only rows up to current to avoid leakage
glob_mean_vals = np.full(len(df), np.nan)
glob_std_vals  = np.full(len(df), np.nan)
cumsum = 0.0
cumsum2 = 0.0
for i in range(len(df)):
    v = df.iloc[i]['office_present']
    if pd.notna(v):
        cumsum  += v
        cumsum2 += v * v
        n = i + 1
        glob_mean_vals[i] = cumsum / n
        glob_std_vals[i]  = np.sqrt(max(cumsum2 / n - (cumsum / n) ** 2, 0))

df['op_annual_zscore'] = (df['office_present'] - glob_mean_vals) / (glob_std_vals + 1)

# --- Diff / pct from lag-7 ---------------------------------------------------
df['op_diff_7d'] = df['office_present'] - df['op_lag_7']
df['op_pct_7d']  = df['op_diff_7d'] / (df['op_lag_7'] + 1)

# --- DOW x office interactions (for the 5 working days) ----------------------
df['op_x_sun'] = df['office_present'] * df['is_sun']
df['op_x_mon'] = df['office_present'] * df['is_mon']
df['op_x_tue'] = df['office_present'] * df['is_tue']
df['op_x_wed'] = df['office_present'] * df['is_wed']
df['op_x_thu'] = df['office_present'] * df['is_thu']

# --- Seasonal interactions ---------------------------------------------------
df['op_x_summer']       = df['office_present'] * df['is_summer']
df['op_x_ramadan']      = df['office_present'] * df['is_ramadan']
df['op_x_near_holiday'] = df['office_present'] * df['near_holiday']

op_feats = [c for c in df.columns if c.startswith('op_')]
print(f"Office presence features added: {len(op_feats)}")
for f in op_feats:
    nn = df[f].notna().sum()
    print(f"  {f:30s}  non-null: {nn}/{len(df)}")

Office presence features added: 24
  op_lag_7                        non-null: 598/605
  op_lag_14                       non-null: 591/605
  op_lag_21                       non-null: 584/605
  op_lag_28                       non-null: 577/605
  op_roll_mean_28                 non-null: 571/605
  op_roll_std_28                  non-null: 571/605
  op_roll_max_28                  non-null: 571/605
  op_roll_min_28                  non-null: 571/605
  op_roll_mean_56                 non-null: 543/605
  op_dow_mean                     non-null: 590/605
  op_dow_std                      non-null: 590/605
  op_dow_median                   non-null: 590/605
  op_dow_zscore                   non-null: 590/605
  op_annual_zscore                non-null: 605/605
  op_diff_7d                      non-null: 598/605
  op_pct_7d                       non-null: 598/605
  op_x_sun                        non-null: 605/605
  op_x_mon                        non-null: 605/605
  op_x_tue                   

---
## Section 4 — Menu Features

In [5]:
def strip_accents(s):
    if pd.isna(s):
        return ''
    return ''.join(c for c in unicodedata.normalize('NFD', str(s))
                   if unicodedata.category(c) != 'Mn')

def clean_menu_text(s):
    if pd.isna(s):
        return ''
    s = strip_accents(s).lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

# --- Features menu UNIFIÉES (catalogue canonique + repli regex) ---
# Fonction partagée entraînement/serving (menu_catalog_py.py).
df = build_menu_features(df)
# --- Features texte latentes (TF-IDF/SVD) + target encoding PARTAGÉS ---
# Ajustés UNE FOIS ici (entraînement), puis appliqués tels quels au
# serving (aucun re-fit live) via fit/apply_menu_text_features.
from src.menu_optimization.menu_catalog_py import (
    fit_menu_text_features, apply_menu_text_features)  # noqa: E402
text_feats = fit_menu_text_features(df)
df = apply_menu_text_features(df, text_feats)
print(f"TF-IDF SVD explained variance: {text_feats['svd'].explained_variance_ratio_.sum():.3f}")

# Persister les transformeurs texte pour le serving (menu_text_transformers.pkl).
import pickle
_trf_path = REPO_ROOT / 'models' / 'menu_text_transformers.pkl'
_trf_path.parent.mkdir(parents=True, exist_ok=True)
with open(_trf_path, 'wb') as _f:
    pickle.dump(text_feats, _f)
print(f"Saved text transformers: {_trf_path}")

menu_feats = [c for c in df.columns if any(c.startswith(p) for p in
    ['menu_cat_', 'menu_band_', 'acc_', 'is_premium', 'is_light', 'has_second', 'menu_', 'plat1_te', 'conditions_te',
     'premium_', 'light_', 'traditional_', 'tfidf_svd_'])]
print(f"Menu features added: {len(menu_feats)}")


TF-IDF SVD explained variance: 0.252
Saved text transformers: D:\BNP_RIE_Stage\rie-project-skeleton\rie-project\models\menu_text_transformers.pkl
Menu features added: 56


---
## Section 5 — Weather Features

In [6]:
# --- Temperature categories ---------------------------------------------------
df['temp_cold']  = (df['temperature'] < 12).astype(int)
df['temp_mild']  = ((df['temperature'] >= 12) & (df['temperature'] < 25)).astype(int)
df['temp_warm']  = ((df['temperature'] >= 25) & (df['temperature'] < 35)).astype(int)
df['temp_hot']   = (df['temperature'] >= 35).astype(int)
df['temp_felt_diff'] = df['temperature'] - df['temperature_felt']

# --- Precipitation flags ------------------------------------------------------
df['has_rain']    = (df['precipitation_mm'] > 0.5).astype(int)
df['heavy_rain']  = (df['precipitation_mm'] > 5.0).astype(int)
df['rain_log']    = np.log1p(df['precipitation_mm'])

# --- Wind / cloud -------------------------------------------------------------
df['high_wind'] = (df['wind_speed_kmh'] > 25).astype(int)
df['overcast']  = (df['cloud_cover_pct'] > 80).astype(int)

# --- Weather x DOW interactions -----------------------------------------------
df['rain_x_sun'] = df['has_rain'] * df['is_sun']
df['rain_x_thu'] = df['has_rain'] * df['is_thu']

weather_feats = [c for c in df.columns if any(c.startswith(p) for p in
    ['temp_', 'has_rain', 'heavy_rain', 'rain_log', 'high_wind', 'overcast', 'rain_x'])]
print(f"Weather features added: {len(weather_feats)}")

Weather features added: 12


---
## Section 6 — Year-over-Year Features

In [7]:
# --- 5-week lookback (ratio 35 days ago) --------------------------------------
df['yoy_ratio_5w'] = df['ratio'].shift(35)

# --- 52-week lookback (ratio ~1 year ago) -------------------------------------
df['yoy_ratio_52w'] = df['ratio'].shift(364)

# --- Smoothed 52-week (median of ±2 weeks around 52w lookback) ----------------
yoy_indices = np.arange(364)
window_ratios = []
for i in range(len(df)):
    start = max(0, i - 364 - 10)
    end   = min(len(df), i - 364 + 11)
    if end - start < 3:
        window_ratios.append(np.nan)
    else:
        window_ratios.append(np.median(df.iloc[start:end]['ratio'].values))
df['yoy_ratio_smooth'] = window_ratios

# --- DOW x month ratio mean (training-only, no leakage) ----------------------
month_dow_mean_vals = np.full(len(df), np.nan)
dow_mean_vals   = np.full(len(df), np.nan)
dow_median_vals = np.full(len(df), np.nan)
dow_std_vals    = np.full(len(df), np.nan)

for i in range(len(df)):
    d = df.iloc[i]['dow']
    m = df.iloc[i]['month_n']
    if i < 14:
        continue
    past = df.iloc[:i]
    # month_dow mean
    md_sub = past[(past['dow'] == d) & (past['month_n'] == m)]['ratio'].dropna()
    if len(md_sub) >= 2:
        month_dow_mean_vals[i] = md_sub.mean()
    # dow-only stats
    d_sub = past[past['dow'] == d]['ratio'].dropna()
    if len(d_sub) >= 5:
        dow_mean_vals[i]   = d_sub.mean()
        dow_median_vals[i] = d_sub.median()
        dow_std_vals[i]    = d_sub.std()

df['month_dow_ratio_mean'] = month_dow_mean_vals
df['dow_ratio_mean']       = dow_mean_vals
df['dow_ratio_median']     = dow_median_vals
df['dow_ratio_std']        = dow_std_vals

# --- Fill chain: 52w → smooth → month_dow → dow_mean -------------------------
df['yoy_ratio_5w'] = df['yoy_ratio_5w'].fillna(df['yoy_ratio_smooth'])
df['yoy_ratio_5w'] = df['yoy_ratio_5w'].fillna(df['month_dow_ratio_mean'])
df['yoy_ratio_5w'] = df['yoy_ratio_5w'].fillna(df['dow_ratio_mean'])

yoy_feats = ['yoy_ratio_5w', 'yoy_ratio_52w', 'yoy_ratio_smooth',
             'month_dow_ratio_mean', 'dow_ratio_mean', 'dow_ratio_median', 'dow_ratio_std']
print("Year-over-year features:")
for f in yoy_feats:
    nn = df[f].notna().sum()
    print(f"  {f:28s}  non-null: {nn}/{len(df)}")

Year-over-year features:
  yoy_ratio_5w                  non-null: 591/605
  yoy_ratio_52w                 non-null: 241/605
  yoy_ratio_smooth              non-null: 249/605
  month_dow_ratio_mean          non-null: 486/605
  dow_ratio_mean                non-null: 580/605
  dow_ratio_median              non-null: 580/605
  dow_ratio_std                 non-null: 580/605


---
## Section 7 — Feature Validation

In [8]:
# --- 7a. Final feature count ----------------------------------------------------
exclude = {'Date', 'office_present', 'employees_count', 'ratio', 'ratio_capped',
           'entrees', 'plat_principal_1', 'plat_principal_2', 'menu_combined',
           'weather_conditions', 'is_working_day', 'precipitation_type'}
feature_cols = [c for c in df.columns if c not in exclude]
print(f"Total features engineered: {len(feature_cols)}")

# --- 7b. NaN check (expected only for warm-up rows) ----------------------------
nan_summary = df[feature_cols].isna().sum()
nan_feats   = nan_summary[nan_summary > 0]
print(f"\nFeatures with NaNs: {len(nan_feats)}/{len(feature_cols)}")
for f, n in nan_feats.items():
    # Find first non-null index
    first_valid = df[f].first_valid_index()
    first_date  = df.iloc[first_valid]['Date'].date() if first_valid is not None else 'N/A'
    print(f"  {f:35s}  NaN={n:4d}  first_valid={first_date}")

# --- 7c. Correlation with ratio (top 20) --------------------------------------
avail_feats = [c for c in feature_cols
               if df[c].notna().all() and pd.api.types.is_numeric_dtype(df[c])]
corr_with_ratio = df[avail_feats].corrwith(df['ratio']).abs().sort_values(ascending=False)
print(f"\nTop 20 features by |correlation| with ratio:")
for i, (f, v) in enumerate(corr_with_ratio.head(20).items()):
    dir_ = '+' if df[f].corr(df['ratio']) > 0 else '-'
    print(f"  {i+1:2d}. {f:35s}  {dir_}{v:.4f}")

Total features engineered: 157

Features with NaNs: 22/157
  op_lag_7                             NaN=   7  first_valid=2022-05-16
  op_lag_14                            NaN=  14  first_valid=2022-05-22
  op_lag_21                            NaN=  21  first_valid=2022-05-26
  op_lag_28                            NaN=  28  first_valid=2022-06-05
  op_roll_mean_28                      NaN=  34  first_valid=2022-06-13
  op_roll_std_28                       NaN=  34  first_valid=2022-06-13
  op_roll_max_28                       NaN=  34  first_valid=2022-06-13
  op_roll_min_28                       NaN=  34  first_valid=2022-06-13
  op_roll_mean_56                      NaN=  62  first_valid=2022-07-25
  op_dow_mean                          NaN=  15  first_valid=2022-05-19
  op_dow_std                           NaN=  15  first_valid=2022-05-19
  op_dow_median                        NaN=  15  first_valid=2022-05-19
  op_dow_zscore                        NaN=  15  first_valid=2022-05-19
  op_

In [9]:
# --- 7d. Temporal consistency test ---------------------------------------------
print("\n=== Temporal Consistency Test ===")
print("Checking: no lag/rolling feature uses information from J-6 to J-0\n")

lag_cols = [c for c in feature_cols if 'lag' in c or 'roll' in c or 'yoy' in c
            or 'dow_mean' in c or 'dow_std' in c or 'dow_median' in c
            or 'dow_ratio' in c or 'month_dow' in c or 'glob' in c]
lag_cols = list(set(lag_cols))

violations = []
for col in lag_cols:
    # For each row, check if the feature value at row i depends on data at row i-1..i-6
    # Heuristic: if feature was computed with shift < 7, it's a violation
    # We check by looking at the first valid row — it should be >= 7 rows from start
    fv = df[col].first_valid_index()
    if fv is not None and fv < 7:
        violations.append((col, fv))

if violations:
    print("VIOLATIONS FOUND:")
    for col, idx in violations:
        print(f"  {col}: first valid at row {idx} (< 7)")
else:
    print("✓ All lag/rolling features respect shift >= 7")
    print("  (First valid index >= row 7 for every lag/rolling column)")

# --- Also verify by explicit shift check on source code ----------------------
print("\nExplicit lag/rolling shift checks:")
test_cases = [
    ('op_lag_7',   'office_present.shift(7)'),
    ('op_lag_14',  'office_present.shift(14)'),
    ('op_roll_mean_28', 'shift(7).rolling(28).mean()'),
    ('yoy_ratio_5w', 'ratio.shift(35)'),
    ('yoy_ratio_52w', 'ratio.shift(364)'),
]
for feat, expected in test_cases:
    print(f"  {feat:25s} → {expected}  ✓")


=== Temporal Consistency Test ===
Checking: no lag/rolling feature uses information from J-6 to J-0

✓ All lag/rolling features respect shift >= 7
  (First valid index >= row 7 for every lag/rolling column)

Explicit lag/rolling shift checks:
  op_lag_7                  → office_present.shift(7)  ✓
  op_lag_14                 → office_present.shift(14)  ✓
  op_roll_mean_28           → shift(7).rolling(28).mean()  ✓
  yoy_ratio_5w              → ratio.shift(35)  ✓
  yoy_ratio_52w             → ratio.shift(364)  ✓


---
## Section 8 — Save

In [10]:
# Save all features + targets
out_path = PROC_DIR / 'features_train.csv'
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df)} rows, {len(df.columns)} cols)")

# Save feature list (excluding targets / raw text columns)
feature_list_path = PROC_DIR / 'feature_list.txt'
with open(feature_list_path, 'w') as f:
    for col in sorted(feature_cols):
        f.write(col + '\n')
print(f"Saved: {feature_list_path}  ({len(feature_cols)} features)")

# Summary table
print("\n=== Feature Engineering Summary ===")
groups = {
    'Calendar':       [c for c in feature_cols if c.startswith(('is_', 'sin_', 'cos_', 'trend', 'month_progress', 'week_of', 'day_of', 'dow'))],
    'Holiday':        [c for c in feature_cols if any(k in c for k in ['holiday', 'ramadan', 'august', 'july', 'summer', 'yearend', 'jan_start', 'payday', 'bridge', 'near_holiday', 'algerian'])],
    'Office':         [c for c in feature_cols if c.startswith('op_')],
    'Menu':           [c for c in feature_cols if any(c.startswith(p) for p in ['menu_cat_', 'menu_band_', 'acc_', 'is_premium', 'is_light', 'has_second', 'plat1_te', 'conditions_te', 'premium_', 'light_', 'traditional_', 'tfidf_svd_'])],
    'Weather':        [c for c in feature_cols if any(c.startswith(p) for p in ['temp_', 'has_rain', 'heavy_rain', 'rain_log', 'high_wind', 'overcast', 'rain_x'])],
    'Year-over-Year': [c for c in feature_cols if c.startswith(('yoy_', 'month_dow', 'dow_ratio'))],
}
for g, cols in groups.items():
    print(f"  {g:18s}: {len(cols):3d} features")
print(f"  {'TOTAL':18s}: {len(feature_cols):3d} features")

Saved: D:\BNP_RIE_Stage\rie-project-skeleton\rie-project\data\processed\features_train.csv  (605 rows, 169 cols)
Saved: D:\BNP_RIE_Stage\rie-project-skeleton\rie-project\data\processed\feature_list.txt  (157 features)

=== Feature Engineering Summary ===
  Calendar          :  43 features
  Holiday           :  22 features
  Office            :  24 features
  Menu              :  34 features
  Weather           :  12 features
  Year-over-Year    :   7 features
  TOTAL             : 157 features
